# Phase Zero Cyber Muscle — Sovereign QLoRA Cloud Trainer

Fine-tunes the **Hybrid Cyber Muscle** on **Google Colab Free Tier (Tesla T4 16GB VRAM)**.
- **Base Model**: `unsloth/DeepSeek-R1-Distill-Qwen-14B-bnb-4bit`
- **Dataset**: 1,540 Cyber Muscle pairs (Phase Zero 4-Pillar Curriculum v1 + KEV/LOLBAS/Kernel) embedded inline
- **Native Pillars**: Protection Reasoning, Deceptive Posture, Invisible Execution, Stealth Architecture
- **Output**: `q8_0` GGUF LoRA adapter auto-downloaded cleanly (SafeTensors ZIP fallback)

**How to run**: `Runtime` -> `Run all` (`Ctrl+F9`).

In [ ]:
# Step 1: Install Unsloth & QLoRA Runtime (~45s)
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes

In [ ]:
# Step 2: Unpack 1,540 Cyber Muscle Training Pairs
import gzip
import base64

CYBER_B64_DATA = "..."
raw_jsonl = gzip.decompress(base64.b64decode(CYBER_B64_DATA)).decode("utf-8")
with open("cyber_muscle_dataset.jsonl", "w", encoding="utf-8") as f:
    f.write(raw_jsonl)

line_count = len(raw_jsonl.strip().splitlines())
print(f"Unpacked {line_count:,} Cyber Muscle pairs ({len(raw_jsonl):,} bytes).")

In [ ]:
# Step 3: Load DeepSeek-R1-Distill-Qwen-14B in 4-bit & Attach LoRA Projections
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/DeepSeek-R1-Distill-Qwen-14B-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)
print("14B model ready with LoRA targets.")

In [ ]:
# Step 4: Format Conversations & Execute QLoRA Training (~12-16 min on Tesla T4)
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments
import torch

dataset = load_dataset("json", data_files="cyber_muscle_dataset.jsonl", split="train")

def format_prompts(batch):
    convos = batch["conversations"]
    texts = []
    has_tpl = hasattr(tokenizer, "apply_chat_template") and tokenizer.chat_template is not None
    for convo in convos:
        msgs = [{"role": ("user" if t.get("from") in ("human", "user") else "assistant"), "content": t.get("value", "")} for t in convo]
        if has_tpl:
            texts.append(tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False))
        else:
            texts.append("\n".join(f"<{m['role']}>\n{m['content']}" for m in msgs))
    return {"text": texts}

dataset = dataset.map(format_prompts, batched=True, remove_columns=dataset.column_names)
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        max_steps=150,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir="outputs",
        report_to="none",
    ),
)

stats = trainer.train()
print(f"Training finished! Final Loss: {stats.training_loss:.4f}")

In [ ]:
# Step 5: Export q8_0 GGUF LoRA Adapter & Download Automatically
import glob
import shutil
from google.colab import files

out_dir = "cyber_muscle_gguf"
try:
    model.save_pretrained_gguf(out_dir, tokenizer, quantization_method="q8_0")
    ggufs = glob.glob(f"{out_dir}/*.gguf")
    if ggufs:
        print(f"Downloading {ggufs[0]}...")
        files.download(ggufs[0])
    else:
        raise RuntimeError("GGUF export produced no .gguf file")
except Exception as exc:
    print(f"GGUF direct fallback ({exc}) -> bundling SafeTensors PEFT adapter...")
    model.save_pretrained("cyber_muscle_adapter")
    shutil.make_archive("cyber_muscle_adapter", "zip", "cyber_muscle_adapter")
    files.download("cyber_muscle_adapter.zip")